# Credit Card Analysis

In [ ]:
## import all libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load the dataset into a dataframe
df = pd.read_csv(
    "https://raw.githubusercontent.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/master/creditcard.csv"
)

In [ ]:
# view data
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

In [ ]:
df["Class"].value_counts()

# The frauds ratio is to less compared to non farud cases

In [ ]:
# check columns with null values
df.isna().sum()[df.isna().sum() > 0]

# there is no column with any null values, data is already cleaned

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 15))
sns.heatmap(df.corr(), cmap="coolwarm", center=0, annot=False)
plt.show()

# If you only care about correlation with Class (more useful given 30 columns — a full heatmap gets unreadable):

corr_with_class = df.corr()["Class"].sort_values(ascending=False)

plt.figure(figsize=(6, 10))
sns.heatmap(corr_with_class.to_frame(), annot=True, cmap="coolwarm", center=0)
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [ ]:
import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=100, max_depth=4, random_state=42, eval_metric="logloss"
)
model.fit(X_train, y_train)

y_predict = model.predict(X_test)


In [ ]:
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
model_spw = xgb.XGBClassifier(
    scale_pos_weight=neg / pos,
    n_estimators=100,
    max_depth=4,
    random_state=42,
    eval_metric="logloss",
)  # XGBoost's built-in weighting

model_spw.fit(X_train, y_train)

y_predict_spw = model_spw.predict(X_test)

In [ ]:
# Calculate metrics for scale_pos_weight xgboost model
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

y_proba_spw = model_spw.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_predict_spw))
print("Precision:", precision_score(y_test, y_predict_spw))
print("Recall:", recall_score(y_test, y_predict_spw))
print("F1 Score:", f1_score(y_test, y_predict_spw))
print("ROC AUC:", roc_auc_score(y_test, y_proba_spw))
print("PR AUC:", average_precision_score(y_test, y_proba_spw))

cm_spw = confusion_matrix(y_test, y_predict_spw)
print(cm_spw)

RocCurveDisplay.from_predictions(y_test, y_proba_spw)
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, y_proba_spw)
plt.show()


In [ ]:
# Calculate metrics for base xgboost model
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

print("Accuracy:", accuracy_score(y_test, y_predict))
print("Precision:", precision_score(y_test, y_predict))
print("Recall:", recall_score(y_test, y_predict))
print("F1 Score:", f1_score(y_test, y_predict))

cm = confusion_matrix(y_test, y_predict)
print(cm)


In [ ]:
# calculate roc and PR AUC
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

y_proba = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)

print("ROC AUC:", roc_auc)
print("PR AUC:", pr_auc)

RocCurveDisplay.from_predictions(y_test, y_proba)
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, y_proba)
plt.show()


In [ ]:
# apply CV on model with n 5
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV

param_dist = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    xgb.XGBClassifier(random_state=42, eval_metric="logloss"),
    param_distributions=param_dist,
    n_iter=20,
    scoring="average_precision",
    cv=cv,
    random_state=42,
    n_jobs=-1,
)

search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV PR AUC:", search.best_score_)

best_model = search.best_estimator_


In [ ]:
# evaluate tuned model on test set
y_predict = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_predict))
print("Precision:", precision_score(y_test, y_predict))
print("Recall:", recall_score(y_test, y_predict))
print("F1 Score:", f1_score(y_test, y_predict))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("PR AUC:", average_precision_score(y_test, y_proba))

print(confusion_matrix(y_test, y_predict))


In [ ]:
# apply CV on model_spw with n 5
param_dist_spw = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "scale_pos_weight": [1, (neg / pos) / 2, neg / pos, (neg / pos) * 2],
}

search_spw = RandomizedSearchCV(
    xgb.XGBClassifier(random_state=42, eval_metric="logloss"),
    param_distributions=param_dist_spw,
    n_iter=20,
    scoring="average_precision",
    cv=cv,
    random_state=42,
    n_jobs=-1,
)

search_spw.fit(X_train, y_train)

print("Best params:", search_spw.best_params_)
print("Best CV PR AUC:", search_spw.best_score_)

best_model_spw = search_spw.best_estimator_


In [ ]:
# evaluate tuned model_spw on test set
y_predict_spw = best_model_spw.predict(X_test)
y_proba_spw = best_model_spw.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_predict_spw))
print("Precision:", precision_score(y_test, y_predict_spw))
print("Recall:", recall_score(y_test, y_predict_spw))
print("F1 Score:", f1_score(y_test, y_predict_spw))
print("ROC AUC:", roc_auc_score(y_test, y_proba_spw))
print("PR AUC:", average_precision_score(y_test, y_proba_spw))

print(confusion_matrix(y_test, y_predict_spw))


In [ ]:
def total_dollar_cost(threshold, probs, y_true, amounts, fp_cost=10.0):
    preds = probs >= threshold
    fn_mask = (y_true == 1) & (preds == 0)  # missed fraud
    fp_mask = (y_true == 0) & (preds == 1)  # false alarm

    fn_cost = amounts[fn_mask].sum()  # a missed fraud costs exactly what was stolen
    fp_cost_total = fp_mask.sum() * fp_cost  # flat review cost per false alarm
    return fn_cost + fp_cost_total


In [ ]:
# sweep thresholds to find the one that minimizes total dollar cost
import numpy as np

amounts_test = X_test["Amount"].values
y_test_arr = y_test.values

thresholds = np.linspace(0.01, 0.99, 99)
costs = [
    total_dollar_cost(t, y_proba_spw, y_test_arr, amounts_test) for t in thresholds
]

best_idx = np.argmin(costs)
best_threshold = thresholds[best_idx]
best_cost = costs[best_idx]

print("Optimal threshold:", best_threshold)
print("Minimum total cost: $", best_cost)

plt.plot(thresholds, costs)
plt.axvline(
    best_threshold, color="red", linestyle="--", label=f"optimal={best_threshold:.2f}"
)
plt.xlabel("Threshold")
plt.ylabel("Total dollar cost")
plt.legend()
plt.show()


In [ ]:
# Brier score and reliability (calibration) curve
from sklearn.metrics import brier_score_loss

brier = brier_score_loss(y_test, y_proba_spw)
print("Brier score:", brier)


In [ ]:
# recalibrate model_spw with isotonic regression (fixes the scale_pos_weight overconfidence)
from sklearn.calibration import CalibratedClassifierCV

calibrated_spw = CalibratedClassifierCV(
    xgb.XGBClassifier(**best_model_spw.get_params()), method="isotonic", cv=5
)
calibrated_spw.fit(X_train, y_train)

y_proba_spw_cal = calibrated_spw.predict_proba(X_test)[:, 1]

print("Brier score (before):", brier_score_loss(y_test, y_proba_spw))
print("Brier score (after):", brier_score_loss(y_test, y_proba_spw_cal))


In [ ]:
# calibration bins as a table, finer resolution in the high-probability tail
calib_df = pd.DataFrame({"y_true": y_test.values, "y_proba": y_proba_spw_cal})

# extra split points near the top (0.9, 0.95, 0.99) instead of even quantiles,
# so the highest-risk tail is not collapsed into one bin
edges = (
    calib_df["y_proba"].quantile([0, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95, 0.99, 1.0]).unique()
)
calib_df["bin"] = pd.cut(calib_df["y_proba"], bins=edges, include_lowest=True)

calib_table = calib_df.groupby("bin", observed=True).agg(
    mean_predicted_prob=("y_proba", "mean"),
    actual_fraud_rate=("y_true", "mean"),
    count=("y_true", "size"),
)

calib_table


In [ ]:
# SHAP: explain individual predictions from the tuned scale_pos_weight model
import shap

explainer = shap.TreeExplainer(best_model_spw)
shap_values = explainer.shap_values(X_test)

# global: which features matter most across all predictions
shap.summary_plot(shap_values, X_test)

# local: why was the single most-confident flagged transaction predicted as fraud?
top_fraud_idx = np.argmax(y_proba_spw)
shap.force_plot(
    explainer.expected_value,
    shap_values[top_fraud_idx],
    X_test.iloc[top_fraud_idx],
    matplotlib=True,
)


In [ ]:
# compare metrics at the default 0.5 cutoff: tuned raw model vs calibrated model
y_predict_spw_cal = calibrated_spw.predict(X_test)

print("--- best_model_spw (raw, threshold=0.5) ---")
print("Accuracy:", accuracy_score(y_test, y_predict_spw))
print("Precision:", precision_score(y_test, y_predict_spw))
print("Recall:", recall_score(y_test, y_predict_spw))
print("F1 Score:", f1_score(y_test, y_predict_spw))
print(confusion_matrix(y_test, y_predict_spw))

print("\n--- calibrated_spw (isotonic, threshold=0.5) ---")
print("Accuracy:", accuracy_score(y_test, y_predict_spw_cal))
print("Precision:", precision_score(y_test, y_predict_spw_cal))
print("Recall:", recall_score(y_test, y_predict_spw_cal))
print("F1 Score:", f1_score(y_test, y_predict_spw_cal))
print(confusion_matrix(y_test, y_predict_spw_cal))


In [ ]:
# compare cost-optimal thresholds: raw (best_model_spw) vs calibrated (calibrated_spw)
def sweep_best_threshold(
    probs, y_true, amounts, thresholds=np.linspace(0.01, 0.99, 99)
):
    costs = [total_dollar_cost(t, probs, y_true, amounts) for t in thresholds]
    best_idx = np.argmin(costs)
    return thresholds[best_idx], costs[best_idx]


raw_threshold, raw_cost = sweep_best_threshold(y_proba_spw, y_test_arr, amounts_test)
cal_threshold, cal_cost = sweep_best_threshold(
    y_proba_spw_cal, y_test_arr, amounts_test
)

print("--- best_model_spw (raw probabilities) ---")
print("Optimal threshold:", raw_threshold)
print("Minimum total cost: $", raw_cost)

print("\n--- calibrated_spw (calibrated probabilities) ---")
print("Optimal threshold:", cal_threshold)
print("Minimum total cost: $", cal_cost)

# metrics at each model's own cost-optimal threshold
y_pred_raw_opt = (y_proba_spw >= raw_threshold).astype(int)
y_pred_cal_opt = (y_proba_spw_cal >= cal_threshold).astype(int)

print("\n--- best_model_spw @ its optimal threshold ---")
print("Precision:", precision_score(y_test, y_pred_raw_opt))
print("Recall:", recall_score(y_test, y_pred_raw_opt))
print("F1 Score:", f1_score(y_test, y_pred_raw_opt))
print(confusion_matrix(y_test, y_pred_raw_opt))

print("\n--- calibrated_spw @ its optimal threshold ---")
print("Precision:", precision_score(y_test, y_pred_cal_opt))
print("Recall:", recall_score(y_test, y_pred_cal_opt))
print("F1 Score:", f1_score(y_test, y_pred_cal_opt))
print(confusion_matrix(y_test, y_pred_cal_opt))
